In [46]:
import os

def loadFAQs(directory_path):
    faqs = {}

    for filename in os.listdir(directory_path):
        #print("directory_path: " + directory_path)
        #print("filename: " + filename)
        #print("\n")
        
        if filename.endswith("faq.txt"):    # assuming FAQs are in .txt files
            file_path = os.path.join(directory_path, filename)

            with open(file_path) as f:
                raw_faq = f.read()    # Only read ANSI format file

            filename_without_ext = os.path.splitext(filename)[0]    #remove .txt extention
            faqs[filename_without_ext] = [text.strip() for text in raw_faq.split('=====')]

            print("directory_path: " + directory_path)
            print("filename: " + filename)
            print("\n")

    return faqs

In [51]:
faqs = loadFAQs('D:/LLM')
faqs

directory_path: D:/LLM
filename: faq.txt




{'faq': ['What is Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier allows you to sign up for an Oracle Cloud \naccount which provides a number of Always Free services and a \nFree Trial with US$300 of free credit to use on all eligible \nOracle Cloud Infrastructure services for up to 30 days. The \nAlways Free services are available for an unlimited period of \ntime. The Free Trial services may be used until your US$300 of \nfree credits are consumed or the 30 days has expired, whichever \ncomes first.',
  'Who should use Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier services are for everyone. Whether you’re \na developer building and testing applications, a startup founder \ncreating new systems with the intention of scaling later, an \nenterprise looking to test things before moving to cloud, a \nstudent wanting to learn, or an academic developing curriculum \nin the cloud, Oracle Cloud Free Tier enables you to learn, \nexplore, build and test for free.',
  'Why do I need 

In [52]:
docs = [{'text': filename + ' | ' + section, 'path': filename} for filename, sections in faqs.items() for section in sections]

# Sample the resulting data
docs[:2]


[{'text': 'faq | What is Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier allows you to sign up for an Oracle Cloud \naccount which provides a number of Always Free services and a \nFree Trial with US$300 of free credit to use on all eligible \nOracle Cloud Infrastructure services for up to 30 days. The \nAlways Free services are available for an unlimited period of \ntime. The Free Trial services may be used until your US$300 of \nfree credits are consumed or the 30 days has expired, whichever \ncomes first.',
  'path': 'faq'},
 {'text': 'faq | Who should use Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier services are for everyone. Whether you’re \na developer building and testing applications, a startup founder \ncreating new systems with the intention of scaling later, an \nenterprise looking to test things before moving to cloud, a \nstudent wanting to learn, or an academic developing curriculum \nin the cloud, Oracle Cloud Free Tier enables you to learn, \nexplore, build

In [56]:
# Connection detailes
un = os.getenv("user", "TESTDB")
pw = os.getenv("password", "?renoma1")
cs = os.getenv("connstr", "localhost:1521/FREEPDB1")

# Connect to the database
import oracledb

connection = oracledb.connect(user=un, password=pw, dsn=cs)

table_name = 'faqs'

with connection.cursor() as cursor:
    # Create the table
    create_table_sql = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id NUMBER PRIMARY KEY,
            payload CLOB CHECK (payload IS JSON),
            vector VECTOR
            )"""
    try:
        cursor.execute(create_table_sql)
    except oracledb.DatabaseError as e:
        raise

    connection.autocommit = True

In [59]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', local_files_only=True)

import array

# Define a list to store the data
data = [
    {"id": idx, "vector_source": row['text'], "payload": row}
    for idx, row in enumerate(docs)
]

# Collect all text for batch encoding
texts = [f"{row['vector_source']}" for row in data]

# Encode all texts in a batch
embeddings = encoder.encode(texts, batch_size=32, show_progress_bar=True)

# Assign the embedding back to your data structure
for row, embedding in zip(data, embeddings):
    row['vector'] = array.array("f", embedding)    

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [60]:
import json

with connection.cursor() as cursor:
    # Truncate the table
    cursor.execute(f"TRUNCATE TABLE {table_name}")

    prepared_data = [(row['id'], json.dumps(row['payload']), row['vector']) for row in data]

    # Insert the data
    cursor.executemany(
        f"""INSERT INTO {table_name} (id, payload, vector)
        VALUES (:1, :2, :3)""",
        prepared_data
    )

    connection.commit()

In [62]:
with connection.cursor() as cursor:
    # Define the query to select all rows from a table
    query = f"""SELECT *
                  FROM {table_name}"""

    # Execute the query
    cursor.execute(query)

    # Fetch all rows
    rows = cursor.fetchall()

    # Print the rows
    for row in rows[:5]:
        print(row)

(0, {'text': 'faq | What is Oracle Cloud Free Tier?  \n \nOracle Cloud Free Tier allows you to sign up for an Oracle Cloud \naccount which provides a number of Always Free services and a \nFree Trial with US$300 of free credit to use on all eligible \nOracle Cloud Infrastructure services for up to 30 days. The \nAlways Free services are available for an unlimited period of \ntime. The Free Trial services may be used until your US$300 of \nfree credits are consumed or the 30 days has expired, whichever \ncomes first.', 'path': 'faq'}, array('f', [-0.10949961096048355, -0.0003804909356404096, 0.033021003007888794, -0.03412816300988197, 0.04885154590010643, 0.032618336379528046, -0.014728215523064137, 0.01595495641231537, 0.0668984055519104, 0.09125178307294846, 0.017057016491889954, -0.08202415704727173, -0.015324089676141739, -0.07889862358570099, 0.11586431413888931, -0.021172236651182175, -0.02558819018304348, -0.05584024265408516, 0.03818408027291298, -0.037633948028087616, -0.013955

In [69]:
# Define the SQL script used to retrive the chunk
topK = 3

sql = f"""SELECT payload, 
                 VECTOR_DISTANCE(vector, :vector, COSINE) as score
            FROM {table_name}
           ORDER BY score
           FETCH APPROX FIRST {topK} ROWS ONLY"""

# Transform the question into a vector
question = "What is a Always Free?"

# Execute the query
with connection.cursor() as cursor:
    embedding = list(encoder.encode(question))
    vector = array.array("f", embedding)

    results = []

    for (info, score) in cursor.execute(sql, vector=vector):
        text_content = info.read()
        results.append((score, json.loads(text_content)))

# Print the reslts
import pprint
pprint.pp(results)

[(0.6147826420320468,
  {'text': 'faq | What is Oracle Cloud Free Tier?  \n'
           ' \n'
           'Oracle Cloud Free Tier allows you to sign up for an Oracle Cloud \n'
           'account which provides a number of Always Free services and a \n'
           'Free Trial with US$300 of free credit to use on all eligible \n'
           'Oracle Cloud Infrastructure services for up to 30 days. The \n'
           'Always Free services are available for an unlimited period of \n'
           'time. The Free Trial services may be used until your US$300 of \n'
           'free credits are consumed or the 30 days has expired, whichever \n'
           'comes first.',
   'path': 'faq'}),
 (0.704836070109877,
  {'text': 'faq | Who should use Oracle Cloud Free Tier?  \n'
           ' \n'
           'Oracle Cloud Free Tier services are for everyone. Whether you’re \n'
           'a developer building and testing applications, a startup founder \n'
           'creating new systems with the intent